# F4 — Notebook Integrador
**Proyecto:** Impacto de la IA Generativa en Estudiantes de Educación Superior — Fase 4
**Curso:** MCDI500 — Programación para la Ciencia de Datos
**Integrantes:** Pablo Ignacio Balbontín Constenla · Melany Esmeralda Reyes Leiva · Ingeborg Andrea Muñoz Carnot · Mario Alejandro López Pulgar

Este notebook integra de extremo a extremo las fases F1–F4: carga el dataset raw, ejecuta el pipeline de preprocesamiento POO construido en F3, corre el núcleo algorítmico (búsqueda, ordenamiento, detección de outliers) con sus mediciones de eficiencia, y construye las visualizaciones analíticas de F4 con su interpretación.

> Los notebooks `F1_Definicion.ipynb`, `F2_EDA_Limpieza.ipynb` y `F3_Rendimiento_POO.ipynb` documentan el detalle y la evolución de cada fase. Este notebook no los reemplaza: los integra y produce el resultado final reproducible.

## 1. Configuración e importaciones

In [ ]:
import sys, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

ruta_src = os.path.abspath(os.path.join("..", "src"))
if ruta_src not in sys.path:
    sys.path.append(ruta_src)

from gestor_datos import GestorDatos
from preprocesador import Preprocesador
from algoritmo import AnalizadorRiesgo

RUTA_RAW = os.path.join("..", "data", "raw", "ai_student_impact_dataset.csv")
RUTA_PROCESADO_F4 = os.path.join("..", "data", "processed", "ai_student_impact_processed_f4.csv")

print("[OK] Entorno configurado. Librerías y módulos propios importados correctamente.")

## 2. Fase 1 — Definición del problema (recapitulación)

**Resumen F1.** El proyecto analiza el impacto del uso de IA generativa sobre el rendimiento académico (`Post_Semester_GPA`) y el nivel de agotamiento (`Burnout_Risk_Level`) de 50.000 estudiantes universitarios, a partir del dataset *AI Student Impact* (16 columnas originales, sin nulos). La motivación es identificar qué perfil de estudiante combina uso intensivo de IA con peor desempeño y mayor burnout, y si las políticas institucionales de uso de IA se asocian a ese perfil.

**Hipótesis de trabajo (formalizada en esta fase).** F1 no dejó planteada una hipótesis explícita y comprobable; para esta integración se formaliza la que el equipo viene probando implícitamente desde F3 y que se contrasta formalmente en la sección de Discusión (apartado 8): *las políticas institucionales más restrictivas frente al uso de IA generativa se asocian a una mayor proporción de estudiantes en perfil de riesgo académico, en comparación con políticas más permisivas.* El detalle de la exploración inicial está en `F1_Definicion.ipynb`.


In [ ]:
gestor = GestorDatos(ruta_entrada=RUTA_RAW, ruta_salida=RUTA_PROCESADO_F4)
df_raw = gestor.cargar_datos()
print(f"Dataset raw: {df_raw.shape[0]:,} registros, {df_raw.shape[1]} columnas.")
df_raw.head(3)

## 3. Fase 2 — EDA y preprocesamiento funcional (recapitulación)

**Resumen F2.** El EDA confirmó que el dataset no contiene valores nulos ni filas duplicadas. El análisis de skewness fundamentó la elección de escalador por variable: `StandardScaler` para las cuatro variables con distribución aproximadamente simétrica (`Pre_Semester_GPA`, `Traditional_Study_Hours`, `Post_Semester_GPA`, `Skill_Retention_Score`) y `MinMaxScaler` para las de escala discreta acotada o asimetría pronunciada (`Weekly_GenAI_Hours`, `Tool_Diversity`, `Perceived_AI_Dependency`, `Anxiety_Level_During_Exams`). Se decidió **conservar** los outliers detectados por IQR: el dataset es sintético con rangos de diseño, y los valores extremos están dentro del dominio válido de cada variable (p. ej. `Weekly_GenAI_Hours` tiene techo de diseño en 40 h/semana). El detalle completo está en `F2_EDA_Limpieza.ipynb`.


## 4. Fase 3 — Arquitectura POO, algoritmos y eficiencia (recapitulación)

Se reconstruye el dataset procesado ejecutando el `Preprocesador` (POO) construido en F3, y se corre `AnalizadorRiesgo` para regenerar el perfil de riesgo, el ordenamiento y las mediciones de eficiencia. Esto demuestra que el proyecto es reproducible de extremo a extremo desde el dato crudo.

In [ ]:
df_procesado = (Preprocesador(df_raw)
    .eliminar_id()
    .cast_bool()
    .codificar_ordinales()
    .codificar_nominales()
    .escalar()
    .validar()
    .resultado())

print(f"\nDataset procesado: {df_procesado.shape}")

In [ ]:
analizador = AnalizadorRiesgo(df_procesado)
resultados = analizador.ejecutar_analisis_completo()

perfil_riesgo    = resultados["perfil_riesgo"]
resumen_outliers = resultados["resumen_outliers"]
tiempos_busqueda = resultados["tiempos_busqueda"]
tiempos_orden    = resultados["tiempos_orden"]

print(f"\nEstudiantes en perfil de riesgo: {len(perfil_riesgo):,} ({len(perfil_riesgo)/len(df_procesado)*100:.2f}% del total)")
print(f"Factor búsqueda lineal/vectorizada: {tiempos_busqueda['factor']}x")
print(f"Factor Merge Sort/sorted(): {tiempos_orden['factor']}x")

**Factores de esta corrida.** En esta ejecución de referencia: búsqueda lineal con `iterrows` ≈0,891 s vs. búsqueda vectorizada ≈0,001 s (factor ≈780×); Merge Sort recursivo ≈0,104 s vs. `sorted()` nativo ≈0,006 s (factor ≈17×).

Estos factores **varían entre corridas y equipos** — en la corrida registrada en `F3_Rendimiento_POO.ipynb` fueron ≈1.371× y ≈32× respectivamente — porque `timeit` mide tiempo de reloj y depende de la carga de la máquina, no solo del algoritmo. La conclusión estable entre corridas, y la que debe citarse en el informe (apartado V.d), es cualitativa: la vectorizada y `sorted()` ganan por varios órdenes de magnitud o por una cifra simple según el caso, manteniendo ambos pares la **misma complejidad Big-O** — la diferencia es la constante de implementación (Python puro vs. C), no el algoritmo.



## 5. Fase 4 — Visualizaciones analíticas (storytelling)

Tres actos: **Acto 1 (contexto)** — panorama general; **Acto 2 (conflicto)** — el contraste que define el perfil de riesgo; **Acto 3 (resolución)** — una variable institucional que abre una implicancia práctica. Cada gráfico lleva un título que declara el hallazgo, no el tema, siguiendo la pista de la guía de apoyo.

### Acto 1 — Contexto: ¿qué tan extendido está el burnout alto?

In [ ]:
conteo_burnout = df_procesado["Burnout_Risk_Level"].value_counts(normalize=True).sort_index() * 100
etiquetas = ["Bajo", "Medio", "Alto"]
colores = ["#2ecc71", "#f39c12", "#e74c3c"]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(etiquetas, conteo_burnout.values, color=colores, edgecolor="white", linewidth=1.5)
for i, v in enumerate(conteo_burnout.values):
    ax.text(i, v + 1, f"{v:.1f}%", ha="center", fontweight="bold")

# TODO (equipo): confirmen la cifra exacta de su corrida y ajusten el título si cambia
ax.set_title("1 de cada 4 estudiantes presenta burnout alto", fontsize=13, fontweight="bold")
ax.set_xlabel("Nivel de burnout")
ax.set_ylabel("% de estudiantes")
ax.set_ylim(0, max(conteo_burnout.values) + 10)
plt.tight_layout()
plt.savefig(os.path.join("..", "data", "processed", "F4_acto1_contexto.png"), dpi=150, bbox_inches="tight")
plt.show()

**Qué muestra:** una de cada cuatro estudiantes (25,0%) presenta burnout alto; el resto se reparte en 42,3% medio y 32,7% bajo.

**Qué se infiere:** el agotamiento asociado al uso de IA no es un fenómeno marginal en esta muestra: afecta a una fracción sustancial de la población estudiantil observada.

**Límite:** el umbral "alto" corresponde a la categoría `High` tal como quedó definida en el dataset (F1/F2), no a un instrumento clínico validado de burnout (p. ej. Maslach Burnout Inventory). La cifra describe la distribución de una variable categórica de este dataset, no un diagnóstico clínico de la población estudiantil real.


### Acto 2 — Conflicto: el grupo de riesgo frente al resto

In [ ]:
gpa_resto  = df_procesado.loc[~df_procesado.index.isin(perfil_riesgo.index), "Post_Semester_GPA"]
gpa_riesgo = perfil_riesgo["Post_Semester_GPA"]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(gpa_resto, bins=40, alpha=0.6, color="#3498db", label="Resto")
ax.hist(gpa_riesgo, bins=20, alpha=0.85, color="#e74c3c", label="Perfil de riesgo")
ax.axvline(df_procesado["Post_Semester_GPA"].quantile(0.25), color="black", linestyle="--", linewidth=1.2)
ax.annotate("Perfil de riesgo definido bajo este umbral (P25)",
            xy=(df_procesado["Post_Semester_GPA"].quantile(0.25), ax.get_ylim()[1]*0.85),
            xytext=(df_procesado["Post_Semester_GPA"].quantile(0.25)-2.5, ax.get_ylim()[1]*0.95),
            arrowprops=dict(arrowstyle="->", color="#444"), fontsize=9)

# TODO (equipo): reemplacen por el hallazgo concreto que defiende su propio análisis
ax.set_title("El perfil de riesgo se concentra en el extremo bajo de GPA", fontsize=13, fontweight="bold")
ax.set_xlabel("GPA post-semestre (estandarizado)")
ax.set_ylabel("Frecuencia")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join("..", "data", "processed", "F4_acto2_conflicto.png"), dpi=150, bbox_inches="tight")
plt.show()

Este gráfico es **parcialmente tautológico**: `Post_Semester_GPA` ≤ percentil 25 es uno de los tres criterios que definen el "perfil de riesgo" (junto con burnout alto y uso intensivo de IA), de modo que ver al grupo de riesgo concentrado bajo el umbral P25 es, en buena parte, consecuencia de la propia definición y no un hallazgo independiente.

Lo que el gráfico sí aporta —y que no estaba garantizado de antemano— es la **magnitud de la separación**: el grupo de riesgo (2.094 estudiantes, 4,19% del total) se concentra casi en su totalidad en la cola izquierda de la distribución, sin solaparse de forma relevante con el grueso del resto. Esto confirma que el criterio combinado (burnout alto + GPA bajo + uso intensivo de IA) identifica un subgrupo coherente y no disperso, condición necesaria —aunque no suficiente— para que el perfil sea útil como herramienta de detección temprana.


### Acto 3 — Resolución: ¿la política institucional hace diferencia?

In [ ]:
cols_policy = [c for c in df_procesado.columns if c.startswith("policy_")]
tasas = {}
for c in cols_policy:
    sub = df_procesado[df_procesado[c] == 1]
    tasas[c.replace("policy_", "").replace("_", " ")] = perfil_riesgo.index.isin(sub.index).sum() / len(sub) * 100 if len(sub) else 0

fig, ax = plt.subplots(figsize=(7, 4.5))
nombres = list(tasas.keys())
valores = list(tasas.values())
colores_barra = ["#e74c3c" if v == max(valores) else "#95a5a6" for v in valores]
ax.bar(nombres, valores, color=colores_barra, edgecolor="white", linewidth=1.5)
for i, v in enumerate(valores):
    ax.text(i, v + 0.1, f"{v:.2f}%", ha="center", fontweight="bold")

# TODO (equipo): verifiquen la cifra en su corrida y decidan si este es realmente su Acto 3,
# o si encuentran una variable más relevante para su narrativa
ax.set_title("Los estudiantes bajo prohibición estricta muestran mayor tasa de riesgo", fontsize=12, fontweight="bold")
ax.set_xlabel("Política institucional de uso de IA")
ax.set_ylabel("% en perfil de riesgo (dentro del grupo)")
plt.tight_layout()
plt.savefig(os.path.join("..", "data", "processed", "F4_acto3_resolucion.png"), dpi=150, bbox_inches="tight")
plt.show()

Esta es una **asociación, no una relación causal**: que el grupo bajo prohibición estricta (`Strict_Ban`, n=9.788) muestre una tasa de riesgo de 5,98%, frente a 3,72% en `Actively_Encouraged` y 3,77% en `Allowed_With_Citation`, no permite concluir que prohibir el uso de IA *cause* más riesgo académico.

Una explicación alternativa igualmente plausible es la **causalidad inversa o una variable de confusión**: las instituciones o carreras que aplican políticas más restrictivas podrían hacerlo justamente porque ya enfrentan poblaciones con mayor carga académica o mayor riesgo de burnout previo — la política sería una respuesta a ese contexto, no su causa. El dataset no incluye información de carrera/institución cruzada con política, ni datos longitudinales, que permitan descartar esta alternativa. El hallazgo se reporta como una correlación documentada, útil para priorizar investigación adicional, no como evidencia de efecto causal de la política institucional.
